[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/02-data-science/10_hypothesis-testing.ipynb)

*Python for Data Science · Notebook 10 of 13*

# Hypothesis Testing & Inference

Decide whether a difference is real or luck: null and alternative hypotheses, p-values, z and t tests, chi-square, ANOVA, non-parametric and permutation tests, the bootstrap, power and sample size, effect sizes and multiple testing.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** A hypothesis test is a courtroom: assume innocence (no effect) and only convict when the evidence would be very surprising otherwise.

**What interviewers really test here:** What a p-value is and isn't, choosing the right test, Type I vs II errors, power and sample size, peeking, and multiple comparisons.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. The logic of a hypothesis test *(Basics)*
- [ ] 2. Two kinds of mistakes *(Basics)*
- [ ] 3. Which test? z, t, paired, Welch *(Core)*
- [ ] 4. Categories and many groups: χ² and ANOVA *(Core)*
- [ ] 5. When formulas don't fit: ranks, permutations, the bootstrap *(Advanced)*
- [ ] 6. Significant is not the same as important *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [ ]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Vocabulary

| Term | Meaning |
| --- | --- |
| H₀ / H₁ | null (no effect) / alternative hypothesis |
| α | significance level: the false-positive rate you accept (0.05) |
| p-value | P(statistic at least this extreme \| H₀ true) |
| Power (1 − β) | P(reject H₀ \| a real effect of a given size exists), target 0.8 |
| Standard error | standard deviation of an estimate: s/√n for a mean, √(p(1−p)/n) for a proportion |
| Degrees of freedom | how many values are free to vary after estimating parameters |
| Confidence interval | estimate ± critical value × SE; 95% of such intervals contain the true value |

### Test statistics

| Test | Statistic | df |
| --- | --- | --- |
| z (one mean) | (x̄ − μ₀) / (σ/√n) | — |
| t (one mean) | (x̄ − μ₀) / (s/√n) | n − 1 |
| Welch t | (x̄₁ − x̄₂) / √(s₁²/n₁ + s₂²/n₂) | Welch–Satterthwaite |
| Two proportions | (p̂₂ − p̂₁) / √(p̂(1−p̂)(1/n₁ + 1/n₂)) | — |
| χ² | Σ (O − E)² / E | (r−1)(c−1) or k − 1 |
| ANOVA F | (SSB/(k−1)) / (SSW/(n−k)) | k − 1, n − k |

### Assumptions to check

- **Independence**: observations don't influence each other (no repeated users counted as separate samples).
- **Normality** of the mean: fine for n ≳ 30 by the central limit theorem; otherwise use ranks, permutations or the bootstrap.
- **Equal variances**: don't assume it — Welch's t-test is safe either way.
- **Expected counts ≥ 5** for χ²; otherwise use Fisher's exact test.

---
# Part 2: Lessons, step by step

## Lesson 1: The logic of a hypothesis test

*Level: Basics*

Your new checkout page converted 12.5% of users; the old one 10%. Is the new page better, or did you just get a lucky week? A hypothesis test answers: **"if there were really no difference, how surprising would this data be?"**




1. **Null hypothesis H₀**: no effect (the rates are equal). **Alternative H₁**: there is an effect.
2. Pick a **significance level α** before looking, usually 0.05.
3. Compute a **test statistic** (z, t, χ², F): how far the data is from H₀, in units of noise.
4. The **p-value** is the probability of a statistic at least this extreme if H₀ were true.
5. If p &lt; α, **reject H₀**. Otherwise you "fail to reject" — which is not proof that H₀ is true.



Think of a courtroom: H₀ is "innocent", the data is the evidence, and α is how much doubt you tolerate before convicting.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [ ]:
from scipy import stats


def p_value_picture():
    z = 1.8
    x = np.linspace(-4, 4, 400)
    fig, ax = plt.subplots(figsize=(5, 2.6))
    ax.plot(x, stats.norm.pdf(x), color="#0f766e")
    ax.fill_between(x, stats.norm.pdf(x), where=abs(x) >= z, color="#f59e0b", alpha=0.6)
    p = 2 * stats.norm.sf(z)
    ax.set_title(f"two-sided p for z = {z}: {p:.3f}")


p_value_picture()

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q3](#q3). Try one now, or carry on and do them in Part 3.

## Lesson 2: Two kinds of mistakes

*Level: Basics*

|   | H₀ really true | H₀ really false |
| --- | --- | --- |
| **Reject H₀** | Type I error (false positive), rate α | correct: you found it (power = 1 − β) |
| **Don't reject** | correct | Type II error (false negative), rate β |



Lowering α makes false alarms rarer but misses more real effects. The way to get both low is **more data**. That's why you decide the sample size before the experiment — and why peeking at the p-value every day and stopping when it dips below 0.05 inflates false positives.

**✏️ Your turn:** you now have what you need for [Q14](#q14), [Q18](#q18). Try one now, or carry on and do them in Part 3.

## Lesson 3: Which test? z, t, paired, Welch

*Level: Core*

| Question | Test | SciPy |
| --- | --- | --- |
| One mean vs a target, σ known | z-test | `norm.sf` |
| One mean vs a target, σ unknown | one-sample t | `ttest_1samp` |
| Two independent groups | Welch's t (default) | `ttest_ind(equal_var=False)` |
| Same people measured twice | paired t | `ttest_rel` |
| Two conversion rates | two-proportion z / 2×2 χ² | `chi2_contingency` |
| 3+ group means | one-way ANOVA | `f_oneway` |
| Two categorical variables | χ² independence | `chi2_contingency` |
| Counts vs expected shares | χ² goodness of fit | `chisquare` |
| Skewed data or outliers | Mann–Whitney U | `mannwhitneyu` |



The t distribution is like the normal curve with fatter tails, because the standard deviation is itself estimated; with 30+ observations they're almost the same. Pairing removes person-to-person variation, so a paired test is far more powerful when it applies.

**✏️ Your turn:** you now have what you need for [Q2](#q2), [Q6](#q6), [Q7](#q7), [Q8](#q8), [Q9](#q9). Try one now, or carry on and do them in Part 3.

## Lesson 4: Categories and many groups: χ² and ANOVA

*Level: Core*

**Chi-square** compares observed counts with the counts you'd expect if H₀ were true: χ² = Σ (O − E)² / E. For a table, E = row total × column total / grand total, and degrees of freedom = (rows − 1)(columns − 1). Expected counts should be at least about 5.



**ANOVA** asks whether any of several means differ. F = variance between group means ÷ variance within groups. A big F says the groups differ more than their internal noise explains. ANOVA only tells you *some* group differs; follow up with pairwise tests (and a correction, below).

**✏️ Your turn:** you now have what you need for [Q10](#q10), [Q11](#q11), [Q12](#q12). Try one now, or carry on and do them in Part 3.

## Lesson 5: When formulas don't fit: ranks, permutations, the bootstrap

*Level: Advanced*

- **Mann–Whitney U** compares ranks, so one huge value can't dominate. Use it for skewed data such as time on site or revenue per user.
- **Permutation test**: if group labels don't matter (H₀), shuffling them shouldn't change the difference much. The p-value is the share of shuffles at least as extreme as what you saw. No distribution assumptions.
- **Bootstrap**: resample your data with replacement thousands of times and recompute the statistic; the middle 95% of those values is a confidence interval. Works for medians, ratios, anything.
- **Exact binomial test**: for small samples of yes/no outcomes, add up the binomial probabilities instead of using the normal approximation.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [ ]:
def bootstrap_picture():
    rng = np.random.default_rng(0)
    data = np.array([12.1, 11.8, 12.6, 12.9, 12.3, 11.5, 12.8, 12.4])
    means = data[rng.integers(0, len(data), size=(2000, len(data)))].mean(axis=1)
    lo, hi = np.percentile(means, [2.5, 97.5])
    fig, ax = plt.subplots(figsize=(5, 2.6))
    ax.hist(means, bins=30, color="#99c9c2")
    ax.axvline(lo, color="#b45309")
    ax.axvline(hi, color="#b45309")
    ax.set_title(f"95% bootstrap CI: {lo:.2f} to {hi:.2f}")


bootstrap_picture()

**✏️ Your turn:** you now have what you need for [Q16](#q16), [Q19](#q19), [Q15](#q15), [Q17](#q17). Try one now, or carry on and do them in Part 3.

## Lesson 6: Significant is not the same as important

*Level: Advanced*

- **Effect size**: with a million users, a 0.01% lift can be "significant" and worthless. Report the size of the effect (difference, lift, Cohen's d ≈ 0.2 small, 0.5 medium, 0.8 large) with a **confidence interval**.
- **Multiple testing**: test 20 metrics at α = 0.05 and you expect one false positive by luck. **Bonferroni** divides α by the number of tests (strict); **Benjamini–Hochberg** controls the share of false discoveries (less strict, common in analytics).
- **Practical checklist** for an A/B test: one primary metric chosen in advance, sample size from a power calculation, randomise users (not sessions), run full weeks, check the split is really 50/50 (sample ratio mismatch), don't peek.

**✏️ Your turn:** you now have what you need for [Q13](#q13), [Q5](#q5), [Q4](#q4), [Q20](#q20). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

**▶ Run this cell before the questions.** It sets up the data and tools they share.

In [ ]:
from scipy import stats

import math


<a id="q1"></a>
### Q1. One-Sample z-Test

🟢 Warm-up · *Known σ · two-sided p*

A machine should fill 500 ml with known σ = `sigma`. Given a sample, return `(z, two-sided p)`, rounded to 4 decimals.

**Examples**

- `sample = [502, 498, 505, 503, 501, 499, 504, 506], mu0 = 500, sigma = 3` → `(2.1213, 0.0339)`
- `sample = [500, 500, 501, 499], mu0 = 500, sigma = 2` → `(0.0, 1.0)`

In [ ]:
def z_test(sample, mu0, sigma):
    """
    A machine should fill 500 ml with known σ = sigma. Given a sample, return
    (z, two-sided p), rounded to 4 decimals.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(z_test, [
    (([502, 498, 505, 503, 501, 499, 504, 506], 500, 3), (2.1213, 0.0339)),
    (([500, 500, 501, 499], 500, 2), (0.0, 1.0)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

z = (x̄ − μ₀) / (σ / √n); the normal CDF from `math.erf`.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`stats.norm.sf(|z|) × 2` — sf is the upper-tail area 1 − CDF.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** z = (x̄ − μ₀) / (σ / √n); the normal CDF from `math.erf`.

```python
def phi(z):
    """Standard normal CDF, from the error function."""
    return 0.5 * (1 + math.erf(z / math.sqrt(2)))


def z_test(sample, mu0, sigma):
    n = len(sample)
    z = (sum(sample) / n - mu0) / (sigma / math.sqrt(n))
    p = 2 * (1 - phi(abs(z)))
    return (round(z, 4), round(p, 4))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `stats.norm.sf(|z|) × 2` — sf is the upper-tail area 1 − CDF.

```python
def z_test(sample, mu0, sigma):
    z = (np.mean(sample) - mu0) / (sigma / np.sqrt(len(sample)))
    return (round(z, 4), round(2 * stats.norm.sf(abs(z)), 4))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: z = (x̄ − μ₀) / (σ / √n); the normal CDF from `math.erf`. That costs O(n) time and O(1) space."

"I can do better: `stats.norm.sf(|z|) × 2` — sf is the upper-tail area 1 − CDF. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/hypothesis-testing/one-sample-z-test)

<a id="q2"></a>
### Q2. One-Sample t Statistic

🟢 Warm-up · *Unknown σ*

Return `(t, degrees of freedom)` testing whether the mean of `sample` equals `mu0`.

**Examples**

- `sample = [12.1, 11.8, 12.6, 12.9, 12.3, 11.5, 12.8, 12.4], mu0 = 12.0` → `(1.753, 7)`
- `sample = [11.2, 11.9, 11.4, 12.0, 11.1, 11.6, 11.8, 11.3], mu0 = 12.0` → `(-3.8726, 7)`

In [ ]:
def t_one(sample, mu0):
    """
    Return (t, degrees of freedom) testing whether the mean of sample equals
    mu0.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(t_one, [
    (([12.1, 11.8, 12.6, 12.9, 12.3, 11.5, 12.8, 12.4], 12.0), (1.753, 7)),
    (([11.2, 11.9, 11.4, 12.0, 11.1, 11.6, 11.8, 11.3], 12.0), (-3.8726, 7)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

t = (x̄ − μ₀) / (s / √n) with the sample std s; df = n − 1.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`stats.ttest_1samp` returns the statistic and its `df`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** t = (x̄ − μ₀) / (s / √n) with the sample std s; df = n − 1.

```python
def mean_var(x):
    m = sum(x) / len(x)
    return m, sum((v - m) ** 2 for v in x) / (len(x) - 1)


def t_one(sample, mu0):
    m, v = mean_var(sample)
    t = (m - mu0) / math.sqrt(v / len(sample))
    return (round(t, 4), len(sample) - 1)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `stats.ttest_1samp` returns the statistic and its `df`.

```python
def t_one(sample, mu0):
    r = stats.ttest_1samp(sample, mu0)
    return (round(float(r.statistic), 4), int(r.df))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: t = (x̄ − μ₀) / (s / √n) with the sample std s; df = n − 1. That costs O(n) time and O(1) space."

"I can do better: `stats.ttest_1samp` returns the statistic and its `df`. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/hypothesis-testing/one-sample-t-statistic)

<a id="q3"></a>
### Q3. One-Tailed or Two-Tailed?

🟢 Warm-up · *Alternative hypotheses*

For a z statistic, return the p-value for `alternative` 'two-sided', 'greater' or 'less'.

**Examples**

- `z = 1.8, alternative = 'two-sided'` → `0.0719`
- `z = 1.8, alternative = 'greater'` → `0.0359`
- `z = 1.8, alternative = 'less'` → `0.9641`

In [ ]:
def tail_p(z, alternative):
    """
    For a z statistic, return the p-value for alternative 'two-sided', 'greater'
    or 'less'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(tail_p, [
    ((1.8, 'two-sided'), 0.0719),
    ((1.8, 'greater'), 0.0359),
    ((1.8, 'less'), 0.9641),
    ((-2.5, 'less'), 0.0062),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Areas under the normal curve from `math.erf`.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`norm.sf` (upper tail) and `norm.cdf` (lower tail).

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Areas under the normal curve from `math.erf`.

```python
def phi(z):
    """Standard normal CDF, from the error function."""
    return 0.5 * (1 + math.erf(z / math.sqrt(2)))


def tail_p(z, alternative):
    if alternative == "greater":
        p = 1 - phi(z)
    elif alternative == "less":
        p = phi(z)
    else:
        p = 2 * (1 - phi(abs(z)))
    return round(p, 4)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `norm.sf` (upper tail) and `norm.cdf` (lower tail).

```python
def tail_p(z, alternative):
    p = {"greater": stats.norm.sf(z), "less": stats.norm.cdf(z), "two-sided": 2 * stats.norm.sf(abs(z))}[alternative]
    return round(float(p), 4)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Areas under the normal curve from `math.erf`. That costs O(1) time and O(1) space."

"I can do better: `norm.sf` (upper tail) and `norm.cdf` (lower tail). That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/hypothesis-testing/one-tailed-or-two-tailed)

<a id="q4"></a>
### Q4. Bonferroni Correction

🟢 Warm-up · *Many tests at once*

You ran several tests. Return the indexes still significant at family-wise α = `alpha` after Bonferroni (compare each p to α / number of tests).

**Examples**

- `pvals = [0.001, 0.02, 0.04, 0.3, 0.008], alpha = 0.05` → `[0, 4]`
- `pvals = [0.03, 0.04], alpha = 0.05` → `[]`

In [ ]:
def bonferroni(pvals, alpha):
    """
    You ran several tests. Return the indexes still significant at family-wise α
    = alpha after Bonferroni (compare each p to α / number of tests).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(bonferroni, [
    (([0.001, 0.02, 0.04, 0.3, 0.008], 0.05), [0, 4]),
    (([0.03, 0.04], 0.05), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop and compare with α/m.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`np.flatnonzero(p < alpha / len(p))`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Loop and compare with α/m.

```python
def bonferroni(pvals, alpha):
    m = len(pvals)
    return [i for i, p in enumerate(pvals) if p < alpha / m]
```

**Cost:** O(m) time, O(m) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `np.flatnonzero(p < alpha / len(p))`.

```python
def bonferroni(pvals, alpha):
    p = np.array(pvals)
    return np.flatnonzero(p < alpha / len(p)).tolist()
```

**Cost:** O(m) time, O(m) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop and compare with α/m. That costs O(m) time and O(m) space."

"I can do better: `np.flatnonzero(p < alpha / len(p))`. That's O(m) time and O(m) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/hypothesis-testing/bonferroni-correction)

<a id="q5"></a>
### Q5. Effect Size: Cohen's d

🟢 Warm-up · *How big, not just whether*

Return Cohen's d = (mean a − mean b) / pooled standard deviation, rounded to 4 decimals.

**Examples**

- `a = [12.1, 11.8, 12.6, 12.9, 12.3, 11.5, 12.8, 12.4], b = [11.2, 11.9, 11.4, 12.0, 11.1, 11.6, 11.8, 11.3]` → `1.8269`
- `a = [10, 12, 14], b = [9, 11, 13]` → `0.5`

In [ ]:
def cohens_d(a, b):
    """
    Return Cohen's d = (mean a − mean b) / pooled standard deviation, rounded to
    4 decimals.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(cohens_d, [
    (([12.1, 11.8, 12.6, 12.9, 12.3, 11.5, 12.8, 12.4], [11.2, 11.9, 11.4, 12.0, 11.1, 11.6, 11.8, 11.3]), 1.8269),
    (([10, 12, 14], [9, 11, 13]), 0.5),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Pool the two sample variances by degrees of freedom.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

NumPy's `var(ddof=1)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Pool the two sample variances by degrees of freedom.

```python
def mean_var(x):
    m = sum(x) / len(x)
    return m, sum((v - m) ** 2 for v in x) / (len(x) - 1)


def cohens_d(a, b):
    m1, v1 = mean_var(a)
    m2, v2 = mean_var(b)
    sp = math.sqrt(((len(a) - 1) * v1 + (len(b) - 1) * v2) / (len(a) + len(b) - 2))
    return round((m1 - m2) / sp, 4)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** NumPy's `var(ddof=1)`.

```python
def cohens_d(a, b):
    a, b = np.array(a), np.array(b)
    sp = np.sqrt(((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2))
    return round((a.mean() - b.mean()) / sp, 4)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Pool the two sample variances by degrees of freedom. That costs O(n) time and O(1) space."

"I can do better: NumPy's `var(ddof=1)`. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/hypothesis-testing/effect-size-cohens-d)

<a id="q6"></a>
### Q6. Welch's t-Test

🟡 Interview standard · *Two groups, unequal variances*

Return `(t, df)` for Welch's two-sample t-test (df rounded to 2 decimals).

**Examples**

- `a = [12.1, 11.8, 12.6, 12.9, 12.3, 11.5, 12.8, 12.4], b = [11.2, 11.9, 11.4, 12.0, 11.1, 11.6, 11.8, 11.3]` → `(3.6538, 12.51)`
- `a = [12.1, 11.8, 12.6, 12.9, 12.3, 11.5, 12.8, 12.4], b = [12.0, 12.2, 11.9]` → `(1.3851, 8.99)`

In [ ]:
def welch(a, b):
    """
    Return (t, df) for Welch's two-sample t-test (df rounded to 2 decimals).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(welch, [
    (([12.1, 11.8, 12.6, 12.9, 12.3, 11.5, 12.8, 12.4], [11.2, 11.9, 11.4, 12.0, 11.1, 11.6, 11.8, 11.3]), (3.6538, 12.51)),
    (([12.1, 11.8, 12.6, 12.9, 12.3, 11.5, 12.8, 12.4], [12.0, 12.2, 11.9]), (1.3851, 8.99)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

t = (x̄₁ − x̄₂)/√(s₁²/n₁ + s₂²/n₂); df from the Welch–Satterthwaite formula.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`stats.ttest_ind(a, b, equal_var=False)` — the safe default.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** t = (x̄₁ − x̄₂)/√(s₁²/n₁ + s₂²/n₂); df from the Welch–Satterthwaite formula.

```python
def mean_var(x):
    m = sum(x) / len(x)
    return m, sum((v - m) ** 2 for v in x) / (len(x) - 1)


def welch(a, b):
    m1, v1 = mean_var(a)
    m2, v2 = mean_var(b)
    se1, se2 = v1 / len(a), v2 / len(b)
    t = (m1 - m2) / math.sqrt(se1 + se2)
    df = (se1 + se2) ** 2 / (se1 ** 2 / (len(a) - 1) + se2 ** 2 / (len(b) - 1))
    return (round(t, 4), round(df, 2))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `stats.ttest_ind(a, b, equal_var=False)` — the safe default.

```python
def welch(a, b):
    r = stats.ttest_ind(a, b, equal_var=False)
    return (round(float(r.statistic), 4), round(float(r.df), 2))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: t = (x̄₁ − x̄₂)/√(s₁²/n₁ + s₂²/n₂); df from the Welch–Satterthwaite formula. That costs O(n) time and O(1) space."

"I can do better: `stats.ttest_ind(a, b, equal_var=False)` — the safe default. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/hypothesis-testing/welchs-t-test)

<a id="q7"></a>
### Q7. Pooled Two-Sample t-Test

🟡 Interview standard · *Equal variances*

Return `(t, df)` for the classic (pooled variance) two-sample t-test.

**Examples**

- `a = [12.1, 11.8, 12.6, 12.9, 12.3, 11.5, 12.8, 12.4], b = [11.2, 11.9, 11.4, 12.0, 11.1, 11.6, 11.8, 11.3]` → `(3.6538, 14)`

In [ ]:
def pooled(a, b):
    """
    Return (t, df) for the classic (pooled variance) two-sample t-test.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(pooled, [
    (([12.1, 11.8, 12.6, 12.9, 12.3, 11.5, 12.8, 12.4], [11.2, 11.9, 11.4, 12.0, 11.1, 11.6, 11.8, 11.3]), (3.6538, 14)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Pool the variances: s² = ((n₁−1)s₁² + (n₂−1)s₂²)/(n₁+n₂−2).

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`stats.ttest_ind(a, b)` (equal\_var=True is its default).

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Pool the variances: s² = ((n₁−1)s₁² + (n₂−1)s₂²)/(n₁+n₂−2).

```python
def mean_var(x):
    m = sum(x) / len(x)
    return m, sum((v - m) ** 2 for v in x) / (len(x) - 1)


def pooled(a, b):
    m1, v1 = mean_var(a)
    m2, v2 = mean_var(b)
    df = len(a) + len(b) - 2
    sp2 = ((len(a) - 1) * v1 + (len(b) - 1) * v2) / df
    t = (m1 - m2) / math.sqrt(sp2 * (1 / len(a) + 1 / len(b)))
    return (round(t, 4), df)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `stats.ttest_ind(a, b)` (equal\_var=True is its default).

```python
def pooled(a, b):
    r = stats.ttest_ind(a, b)
    return (round(float(r.statistic), 4), int(r.df))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Pool the variances: s² = ((n₁−1)s₁² + (n₂−1)s₂²)/(n₁+n₂−2). That costs O(n) time and O(1) space."

"I can do better: `stats.ttest_ind(a, b)` (equal\_var=True is its default). That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/hypothesis-testing/pooled-two-sample-t-test)

<a id="q8"></a>
### Q8. Paired t-Test

🟡 Interview standard · *Before and after*

The same 6 users were timed before and after a redesign. Return `(t, two-sided p)`.

**Examples**

- `before = [31, 28, 35, 40, 30, 33], after = [27, 27, 30, 36, 29, 30]` → `(4.3916, 0.0071)`

In [ ]:
def paired(before, after):
    """
    The same 6 users were timed before and after a redesign. Return (t, two-
    sided p).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(paired, [
    (([31, 28, 35, 40, 30, 33], [27, 27, 30, 36, 29, 30]), (4.3916, 0.0071)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Take the differences and run a one-sample t-test on them against 0 (p from SciPy's t distribution).

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`stats.ttest_rel(before, after)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Take the differences and run a one-sample t-test on them against 0 (p from SciPy's t distribution).

```python
def mean_var(x):
    m = sum(x) / len(x)
    return m, sum((v - m) ** 2 for v in x) / (len(x) - 1)


def paired(before, after):
    d = [x - y for x, y in zip(before, after)]
    m, v = mean_var(d)
    t = m / math.sqrt(v / len(d))
    p = 2 * stats.t.sf(abs(t), len(d) - 1)
    return (round(t, 4), round(float(p), 4))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `stats.ttest_rel(before, after)`.

```python
def paired(before, after):
    r = stats.ttest_rel(before, after)
    return (round(float(r.statistic), 4), round(float(r.pvalue), 4))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Take the differences and run a one-sample t-test on them against 0 (p from SciPy's t distribution). That costs O(n) time and O(n) space."

"I can do better: `stats.ttest_rel(before, after)`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/hypothesis-testing/paired-t-test)

<a id="q9"></a>
### Q9. A/B Test on Conversion Rates

🟡 Interview standard · *Two-proportion z-test*

Control converted `c1` of `n1`, the variant `c2` of `n2`. Return `(z, two-sided p)`.

**Examples**

- `c1 = 200, n1 = 2000, c2 = 250, n2 = 2000` → `(2.502, 0.0124)`
- `c1 = 48, n1 = 500, c2 = 52, n2 = 500` → `(0.4216, 0.6733)`

In [ ]:
def ab_test(c1, n1, c2, n2):
    """
    Control converted c1 of n1, the variant c2 of n2. Return (z, two-sided p).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(ab_test, [
    ((200, 2000, 250, 2000), (2.502, 0.0124)),
    ((48, 500, 52, 500), (0.4216, 0.6733)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Pooled rate p̂, SE = √(p̂(1−p̂)(1/n₁+1/n₂)), z = (p₂ − p₁)/SE.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A 2×2 chi-square without continuity correction is the same test: χ² = z².

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Pooled rate p̂, SE = √(p̂(1−p̂)(1/n₁+1/n₂)), z = (p₂ − p₁)/SE.

```python
def phi(z):
    """Standard normal CDF, from the error function."""
    return 0.5 * (1 + math.erf(z / math.sqrt(2)))


def ab_test(c1, n1, c2, n2):
    p1, p2 = c1 / n1, c2 / n2
    p = (c1 + c2) / (n1 + n2)
    se = math.sqrt(p * (1 - p) * (1 / n1 + 1 / n2))
    z = (p2 - p1) / se
    return (round(z, 4), round(2 * (1 - phi(abs(z))), 4))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** A 2×2 chi-square without continuity correction is the same test: χ² = z².

```python
def ab_test(c1, n1, c2, n2):
    table = [[c1, n1 - c1], [c2, n2 - c2]]
    chi2, p, dof, _ = stats.chi2_contingency(table, correction=False)
    z = np.sign(c2 / n2 - c1 / n1) * np.sqrt(chi2)
    return (round(float(z), 4), round(float(p), 4))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Pooled rate p̂, SE = √(p̂(1−p̂)(1/n₁+1/n₂)), z = (p₂ − p₁)/SE. That costs O(1) time and O(1) space."

"I can do better: A 2×2 chi-square without continuity correction is the same test: χ² = z². That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/hypothesis-testing/a-b-test-on-conversion-rates)

<a id="q10"></a>
### Q10. Is the Die Fair?

🟡 Interview standard · *Chi-square goodness of fit*

Given counts of faces 1–6, return `(χ² statistic, reject fairness at 5%)`.

**Examples**

- `counts = [16, 18, 16, 14, 12, 44]` → `(35.6, True)`
- `counts = [20, 22, 17, 18, 19, 24]` → `(1.7, False)`

In [ ]:
def fair_die(counts):
    """
    Given counts of faces 1–6, return (χ² statistic, reject fairness at 5%).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(fair_die, [
    (([16, 18, 16, 14, 12, 44],), (35.6, True)),
    (([20, 22, 17, 18, 19, 24],), (1.7, False)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Σ (O − E)²/E with E = total/6; compare to the table value 11.07 for 5 degrees of freedom.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`stats.chisquare(observed)` gives the p-value directly.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Σ (O − E)²/E with E = total/6; compare to the table value 11.07 for 5 degrees of freedom.

```python
def fair_die(counts):
    e = sum(counts) / len(counts)
    chi2 = sum((o - e) ** 2 / e for o in counts)
    return (round(chi2, 4), chi2 > 11.0705)
```

**Cost:** O(k) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `stats.chisquare(observed)` gives the p-value directly.

```python
def fair_die(counts):
    r = stats.chisquare(counts)
    return (round(float(r.statistic), 4), bool(r.pvalue < 0.05))
```

**Cost:** O(k) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Σ (O − E)²/E with E = total/6; compare to the table value 11.07 for 5 degrees of freedom. That costs O(k) time and O(1) space."

"I can do better: `stats.chisquare(observed)` gives the p-value directly. That's O(k) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/hypothesis-testing/is-the-die-fair)

<a id="q11"></a>
### Q11. Are Two Categories Related?

🟡 Interview standard · *Chi-square independence*

`table` counts people by group (rows) and choice (columns). Return `(χ², degrees of freedom, expected count in the top-left cell)`.

**Examples**

- `table = [[30, 20, 10], [20, 30, 40]]` → `(16.6667, 2, 20.0)`
- `table = [[10, 10], [10, 10]]` → `(0.0, 1, 10.0)`

In [ ]:
def independence(table):
    """
    table counts people by group (rows) and choice (columns). Return (χ²,
    degrees of freedom, expected count in the top-left cell).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(independence, [
    (([[30, 20, 10], [20, 30, 40]],), (16.6667, 2, 20.0)),
    (([[10, 10], [10, 10]],), (0.0, 1, 10.0)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Expected = row total × column total / grand total; sum (O − E)²/E; df = (r−1)(c−1).

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`stats.chi2_contingency(table, correction=False)` returns all of these.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Expected = row total × column total / grand total; sum (O − E)²/E; df = (r−1)(c−1).

```python
def independence(table):
    rows = [sum(r) for r in table]
    cols = [sum(c) for c in zip(*table)]
    total = sum(rows)
    chi2 = 0
    for i, r in enumerate(table):
        for j, o in enumerate(r):
            e = rows[i] * cols[j] / total
            chi2 += (o - e) ** 2 / e
    return (round(chi2, 4), (len(rows) - 1) * (len(cols) - 1), round(rows[0] * cols[0] / total, 4))
```

**Cost:** O(rc) time, O(rc) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `stats.chi2_contingency(table, correction=False)` returns all of these.

```python
def independence(table):
    chi2, p, dof, expected = stats.chi2_contingency(table, correction=False)
    return (round(float(chi2), 4), int(dof), round(float(expected[0][0]), 4))
```

**Cost:** O(rc) time, O(rc) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Expected = row total × column total / grand total; sum (O − E)²/E; df = (r−1)(c−1). That costs O(rc) time and O(rc) space."

"I can do better: `stats.chi2_contingency(table, correction=False)` returns all of these. That's O(rc) time and O(rc) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/hypothesis-testing/are-two-categories-related)

<a id="q12"></a>
### Q12. One-Way ANOVA

🟡 Interview standard · *Three or more means*

Return the F statistic comparing the means of all the groups.

**Examples**

- `groups = [[85, 86, 88, 75, 78], [91, 92, 93, 85, 87], [79, 78, 88, 94, 92]]` → `2.0`
- `groups = [[12.1, 11.8, 12.6, 12.9, 12.3, 11.5, 12.8, 12.4], [11.2, 11.9, 11.4, 12.0, 11.1, 11.6, 11.8, 11.3]]` → `13.3506`

In [ ]:
def anova_f(groups):
    """
    Return the F statistic comparing the means of all the groups.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(anova_f, [
    (([[85, 86, 88, 75, 78], [91, 92, 93, 85, 87], [79, 78, 88, 94, 92]],), 2.0),
    (([[12.1, 11.8, 12.6, 12.9, 12.3, 11.5, 12.8, 12.4], [11.2, 11.9, 11.4, 12.0, 11.1, 11.6, 11.8, 11.3]],), 13.3506),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

F = (between-group variance) / (within-group variance) from sums of squares.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`stats.f_oneway(*groups)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** F = (between-group variance) / (within-group variance) from sums of squares.

```python
def anova_f(groups):
    allv = [v for g in groups for v in g]
    grand = sum(allv) / len(allv)
    ssb = sum(len(g) * (sum(g) / len(g) - grand) ** 2 for g in groups)
    ssw = sum((v - sum(g) / len(g)) ** 2 for g in groups for v in g)
    k, n = len(groups), len(allv)
    return round((ssb / (k - 1)) / (ssw / (n - k)), 4)
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `stats.f_oneway(*groups)`.

```python
def anova_f(groups):
    return round(float(stats.f_oneway(*groups).statistic), 4)
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: F = (between-group variance) / (within-group variance) from sums of squares. That costs O(n) time and O(k) space."

"I can do better: `stats.f_oneway(*groups)`. That's O(n) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/hypothesis-testing/one-way-anova)

<a id="q13"></a>
### Q13. Confidence Interval for a Difference

🟡 Interview standard · *Large-sample CI*

Return the 95% confidence interval `(low, high)` for mean(a) − mean(b), using the normal critical value and Welch's standard error.

**Examples**

- `a = [12.1, 11.8, 12.6, 12.9, 12.3, 11.5, 12.8, 12.4], b = [11.2, 11.9, 11.4, 12.0, 11.1, 11.6, 11.8, 11.3]` → `(0.3535, 1.1715)`

In [ ]:
def ci_diff(a, b):
    """
    Return the 95% confidence interval (low, high) for mean(a) − mean(b), using
    the normal critical value and Welch's standard error.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(ci_diff, [
    (([12.1, 11.8, 12.6, 12.9, 12.3, 11.5, 12.8, 12.4], [11.2, 11.9, 11.4, 12.0, 11.1, 11.6, 11.8, 11.3]), (0.3535, 1.1715)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

diff ± 1.959964 × √(s₁²/n₁ + s₂²/n₂).

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`stats.norm.ppf(0.975)` for the critical value, NumPy for the rest.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** diff ± 1.959964 × √(s₁²/n₁ + s₂²/n₂).

```python
def mean_var(x):
    m = sum(x) / len(x)
    return m, sum((v - m) ** 2 for v in x) / (len(x) - 1)


def ci_diff(a, b):
    m1, v1 = mean_var(a)
    m2, v2 = mean_var(b)
    se = math.sqrt(v1 / len(a) + v2 / len(b))
    d = m1 - m2
    return (round(d - 1.959964 * se, 4), round(d + 1.959964 * se, 4))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `stats.norm.ppf(0.975)` for the critical value, NumPy for the rest.

```python
def ci_diff(a, b):
    a, b = np.array(a), np.array(b)
    se = np.sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b))
    z = stats.norm.ppf(0.975)
    d = a.mean() - b.mean()
    return (round(d - z * se, 4), round(d + z * se, 4))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: diff ± 1.959964 × √(s₁²/n₁ + s₂²/n₂). That costs O(n) time and O(1) space."

"I can do better: `stats.norm.ppf(0.975)` for the critical value, NumPy for the rest. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/hypothesis-testing/confidence-interval-for-a-difference)

<a id="q14"></a>
### Q14. Power of a z-Test

🟡 Interview standard · *Chance of catching a real effect*

A true difference `delta` exists, σ = `sigma`, n per test, two-sided α = 0.05. Return the power (ignore the tiny opposite tail).

**Examples**

- `delta = 2, sigma = 5, n = 50` → `0.8074`
- `delta = 1, sigma = 5, n = 50` → `0.2926`
- `delta = 1, sigma = 5, n = 400` → `0.9793`

In [ ]:
def power(delta, sigma, n):
    """
    A true difference delta exists, σ = sigma, n per test, two-sided α = 0.05.
    Return the power (ignore the tiny opposite tail).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(power, [
    ((2, 5, 50), 0.8074),
    ((1, 5, 50), 0.2926),
    ((1, 5, 400), 0.9793),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Power = Φ(δ√n/σ − 1.959964).

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`norm.cdf(effect − norm.ppf(0.975))`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Power = Φ(δ√n/σ − 1.959964).

```python
def phi(z):
    """Standard normal CDF, from the error function."""
    return 0.5 * (1 + math.erf(z / math.sqrt(2)))


def power(delta, sigma, n):
    return round(phi(delta * math.sqrt(n) / sigma - 1.959964), 4)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `norm.cdf(effect − norm.ppf(0.975))`.

```python
def power(delta, sigma, n):
    return round(float(stats.norm.cdf(delta * np.sqrt(n) / sigma - stats.norm.ppf(0.975))), 4)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Power = Φ(δ√n/σ − 1.959964). That costs O(1) time and O(1) space."

"I can do better: `norm.cdf(effect − norm.ppf(0.975))`. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/hypothesis-testing/power-of-a-z-test)

<a id="q15"></a>
### Q15. Bootstrap Confidence Interval

🟡 Interview standard · *Resampling*

Resample `data` with replacement 2,000 times (seed 0, NumPy `default_rng`), take each mean, and return the 2.5th and 97.5th percentiles.

**Examples**

- `data = [12.1, 11.8, 12.6, 12.9, 12.3, 11.5, 12.8, 12.4]` → `(11.9875, 12.6)`

In [ ]:
def bootstrap_ci(data):
    """
    Resample data with replacement 2,000 times (seed 0, NumPy default_rng), take
    each mean, and return the 2.5th and 97.5th percentiles.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(bootstrap_ci, [
    (([12.1, 11.8, 12.6, 12.9, 12.3, 11.5, 12.8, 12.4],), (11.9875, 12.6)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Draw the same random row numbers, then average each resample in a plain loop, sort, and interpolate the percentiles by hand.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Draw all resamples as one 2,000 × n matrix and use `np.percentile`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Draw the same random row numbers, then average each resample in a plain loop, sort, and interpolate the percentiles by hand.

```python
def bootstrap_ci(data):
    rng = np.random.default_rng(0)
    idx = rng.integers(0, len(data), size=(2000, len(data)))
    means = sorted(sum(data[i] for i in row) / len(row) for row in idx.tolist())

    def pct(q):
        pos = (len(means) - 1) * q
        lo = int(pos)
        return means[lo] + (means[min(lo + 1, len(means) - 1)] - means[lo]) * (pos - lo)
    return (round(pct(0.025), 4), round(pct(0.975), 4))
```

**Cost:** O(Bn) time, O(B) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** Draw all resamples as one 2,000 × n matrix and use `np.percentile`.

```python
def bootstrap_ci(data):
    rng = np.random.default_rng(0)
    x = np.array(data)
    means = x[rng.integers(0, len(x), size=(2000, len(x)))].mean(axis=1)
    lo, hi = np.percentile(means, [2.5, 97.5])
    return (round(lo, 4), round(hi, 4))
```

**Cost:** O(Bn) time, O(Bn) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Draw the same random row numbers, then average each resample in a plain loop, sort, and interpolate the percentiles by hand. That costs O(Bn) time and O(B) space."

"I can do better: Draw all resamples as one 2,000 × n matrix and use `np.percentile`. That's O(Bn) time and O(Bn) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/hypothesis-testing/bootstrap-confidence-interval)

<a id="q16"></a>
### Q16. Mann–Whitney U

🟡 Interview standard · *Ranks instead of means*

Return the U statistic for `a`: the number of (a, b) pairs where the a value is bigger, counting ties as ½.

**Examples**

- `a = [3, 5, 8, 9], b = [1, 2, 5, 4]` → `13.5`
- `a = [12.1, 11.8, 12.6, 12.9, 12.3, 11.5, 12.8, 12.4], b = [11.2, 11.9, 11.4, 12.0, 11.1, 11.6, 11.8, 11.3]` → `57.5`

In [ ]:
def mwu(a, b):
    """
    Return the U statistic for a: the number of (a, b) pairs where the a value
    is bigger, counting ties as ½.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(mwu, [
    (([3, 5, 8, 9], [1, 2, 5, 4]), 13.5),
    (([12.1, 11.8, 12.6, 12.9, 12.3, 11.5, 12.8, 12.4], [11.2, 11.9, 11.4, 12.0, 11.1, 11.6, 11.8, 11.3]), 57.5),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compare every pair.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`stats.mannwhitneyu(a, b).statistic` — works from ranks, robust to outliers and skew.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Compare every pair.

```python
def mwu(a, b):
    u = 0
    for x in a:
        for y in b:
            u += 1 if x > y else 0.5 if x == y else 0
    return float(u)
```

**Cost:** O(nm) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `stats.mannwhitneyu(a, b).statistic` — works from ranks, robust to outliers and skew.

```python
def mwu(a, b):
    return float(stats.mannwhitneyu(a, b).statistic)
```

**Cost:** O((n+m) log(n+m)) time, O(n+m) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compare every pair. That costs O(nm) time and O(1) space."

"I can do better: `stats.mannwhitneyu(a, b).statistic` — works from ranks, robust to outliers and skew. That's O((n+m) log(n+m)) time and O(n+m) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/hypothesis-testing/mann-whitney-u)

<a id="q17"></a>
### Q17. Exact Binomial Test

🟡 Interview standard · *Small samples, one proportion*

A coin landed heads `k` times in `n` flips. Return the one-sided p-value P(X ≥ k) if the coin is fair (p = `p`).

**Examples**

- `k = 8, n = 10, p = 0.5` → `0.0547`
- `k = 60, n = 100, p = 0.5` → `0.0284`
- `k = 3, n = 20, p = 0.1` → `0.3231`

In [ ]:
def binom_p(k, n, p):
    """
    A coin landed heads k times in n flips. Return the one-sided p-value P(X ≥
    k) if the coin is fair (p = p).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(binom_p, [
    ((8, 10, 0.5), 0.0547),
    ((60, 100, 0.5), 0.0284),
    ((3, 20, 0.1), 0.3231),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Add up C(n, i) pⁱ (1−p)ⁿ⁻ⁱ for i = k … n.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`stats.binomtest(k, n, p, alternative='greater')`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Add up C(n, i) pⁱ (1−p)ⁿ⁻ⁱ for i = k … n.

```python
def binom_p(k, n, p):
    total = 0.0
    for i in range(k, n + 1):
        total += math.comb(n, i) * p ** i * (1 - p) ** (n - i)
    return round(total, 4)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `stats.binomtest(k, n, p, alternative='greater')`.

```python
def binom_p(k, n, p):
    return round(float(stats.binomtest(k, n, p, alternative="greater").pvalue), 4)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Add up C(n, i) pⁱ (1−p)ⁿ⁻ⁱ for i = k … n. That costs O(n) time and O(1) space."

"I can do better: `stats.binomtest(k, n, p, alternative='greater')`. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/hypothesis-testing/exact-binomial-test)

<a id="q18"></a>
### Q18. How Many Users Does the A/B Test Need?

🔴 Stretch · *Sample size*

Return the users needed **per group** to detect a change from rate `p1` to `p2` with two-sided α = `alpha` and power `power` (round up).

**Examples**

- `p1 = 0.1, p2 = 0.12, alpha = 0.05, power = 0.8` → `3839`
- `p1 = 0.3, p2 = 0.33, alpha = 0.05, power = 0.9` → `5034`

In [ ]:
def sample_size(p1, p2, alpha, power):
    """
    Return the users needed per group to detect a change from rate p1 to p2 with
    two-sided α = alpha and power power (round up).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(sample_size, [
    ((0.1, 0.12, 0.05, 0.8), 3839),
    ((0.3, 0.33, 0.05, 0.9), 5034),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

n = (z₁₋α/₂ + z\_power)² (p₁(1−p₁) + p₂(1−p₂)) / (p₁ − p₂)², with z found by bisection on the normal CDF.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`stats.norm.ppf` gives the z values directly.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** n = (z₁₋α/₂ + z\_power)² (p₁(1−p₁) + p₂(1−p₂)) / (p₁ − p₂)², with z found by bisection on the normal CDF.

```python
def phi(z):
    """Standard normal CDF, from the error function."""
    return 0.5 * (1 + math.erf(z / math.sqrt(2)))


def sample_size(p1, p2, alpha, power):
    def inv_phi(q):
        lo, hi = -10.0, 10.0
        for _ in range(100):
            mid = (lo + hi) / 2
            if phi(mid) < q:
                lo = mid
            else:
                hi = mid
        return (lo + hi) / 2
    z = inv_phi(1 - alpha / 2) + inv_phi(power)
    return math.ceil(z ** 2 * (p1 * (1 - p1) + p2 * (1 - p2)) / (p1 - p2) ** 2)
```

**Cost:** O(log 1/ε) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `stats.norm.ppf` gives the z values directly.

```python
def sample_size(p1, p2, alpha, power):
    z = stats.norm.ppf(1 - alpha / 2) + stats.norm.ppf(power)
    return int(np.ceil(z ** 2 * (p1 * (1 - p1) + p2 * (1 - p2)) / (p1 - p2) ** 2))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: n = (z₁₋α/₂ + z\_power)² (p₁(1−p₁) + p₂(1−p₂)) / (p₁ − p₂)², with z found by bisection on the normal CDF. That costs O(log 1/ε) time and O(1) space."

"I can do better: `stats.norm.ppf` gives the z values directly. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/hypothesis-testing/how-many-users-does-the-a-b-test-need)

<a id="q19"></a>
### Q19. Permutation Test

🔴 Stretch · *No formula needed*

Is the difference in means between `a` and `b` real? Shuffle the group labels in every possible way and return the two-sided p-value (twice the smaller one-sided p, at most 1).

**Examples**

- `a = [12, 15, 14, 17], b = [10, 11, 13, 9]` → `0.0571`
- `a = [5, 6, 7], b = [5, 6, 8]` → `1.0`

In [ ]:
from itertools import combinations

def perm_test(a, b):
    """
    Is the difference in means between a and b real? Shuffle the group labels in
    every possible way and return the two-sided p-value (twice the smaller one-
    sided p, at most 1).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(perm_test, [
    (([12, 15, 14, 17], [10, 11, 13, 9]), 0.0571),
    (([5, 6, 7], [5, 6, 8]), 1.0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try every way to choose which values form group a; count how often the difference is at least as extreme.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`stats.permutation_test` with `n_resamples=np.inf` enumerates every split exactly.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Try every way to choose which values form group a; count how often the difference is at least as extreme.

```python
def perm_test(a, b):
    pool = a + b
    observed = sum(a) / len(a) - sum(b) / len(b)
    ge = le = total = 0
    for idx in combinations(range(len(pool)), len(a)):
        ga = [pool[i] for i in idx]
        gb = [pool[i] for i in range(len(pool)) if i not in idx]
        d = sum(ga) / len(ga) - sum(gb) / len(gb)
        total += 1
        ge += d >= observed - 1e-12
        le += d <= observed + 1e-12
    return round(min(1.0, 2 * min(ge, le) / total), 4)
```

**Cost:** O(C(n, k)) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `stats.permutation_test` with `n_resamples=np.inf` enumerates every split exactly.

```python
def perm_test(a, b):
    diff = lambda x, y: np.mean(x) - np.mean(y)
    r = stats.permutation_test((a, b), diff, permutation_type="independent", n_resamples=np.inf)
    return round(float(r.pvalue), 4)
```

**Cost:** O(C(n, k)) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try every way to choose which values form group a; count how often the difference is at least as extreme. That costs O(C(n, k)) time and O(n) space."

"I can do better: `stats.permutation_test` with `n_resamples=np.inf` enumerates every split exactly. That's O(C(n, k)) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/hypothesis-testing/permutation-test)

<a id="q20"></a>
### Q20. False Discovery Rate (Benjamini–Hochberg)

🔴 Stretch · *Adjusted p-values*

Return BH-adjusted p-values (rounded to 4 decimals) in the original order.

**Examples**

- `pvals = [0.001, 0.02, 0.04, 0.3, 0.008]` → `[0.005, 0.0333, 0.05, 0.3, 0.02]`
- `pvals = [0.01, 0.01, 0.5]` → `[0.015, 0.015, 0.5]`

In [ ]:
def bh(pvals):
    """
    Return BH-adjusted p-values (rounded to 4 decimals) in the original order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(bh, [
    (([0.001, 0.02, 0.04, 0.3, 0.008],), [0.005, 0.0333, 0.05, 0.3, 0.02]),
    (([0.01, 0.01, 0.5],), [0.015, 0.015, 0.5]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Sort; adjusted = p × m / rank; then a running minimum from the largest rank down; cap at 1.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`stats.false_discovery_control(p)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Sort; adjusted = p × m / rank; then a running minimum from the largest rank down; cap at 1.

```python
def bh(pvals):
    m = len(pvals)
    order = sorted(range(m), key=lambda i: pvals[i])
    adj = [0.0] * m
    running = 1.0
    for rank in range(m, 0, -1):
        i = order[rank - 1]
        running = min(running, pvals[i] * m / rank)
        adj[i] = round(running, 4)
    return adj
```

**Cost:** O(m log m) time, O(m) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `stats.false_discovery_control(p)`.

```python
def bh(pvals):
    return np.round(stats.false_discovery_control(pvals), 4).tolist()
```

**Cost:** O(m log m) time, O(m) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Sort; adjusted = p × m / rank; then a running minimum from the largest rank down; cap at 1. That costs O(m log m) time and O(m) space."

"I can do better: `stats.false_discovery_control(p)`. That's O(m log m) time and O(m) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/hypothesis-testing/false-discovery-rate-benjamini-hochberg)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. What does a p-value of 0.03 mean?</b></summary>

If there were truly no effect, data at least this extreme would occur 3% of the time. It is not the probability that H₀ is true, and not the probability the result is a fluke.

</details>

<details>
<summary><b>2. Why not stop the A/B test as soon as p &lt; 0.05?</b></summary>

Checking repeatedly gives many chances for noise to cross the line, so the real false-positive rate becomes much higher than 5%. Fix the sample size in advance, or use a sequential method designed for peeking.

</details>

<details>
<summary><b>3. Type I vs Type II error?</b></summary>

Type I: rejecting a true H₀ (false alarm, rate α). Type II: missing a real effect (rate β). More data reduces Type II without raising Type I.

</details>

<details>
<summary><b>4. When use a paired test?</b></summary>

When each observation in one group has a natural partner in the other: the same user before and after, the same product in two stores. Pairing removes between-subject variation.

</details>

<details>
<summary><b>5. t-test vs Mann–Whitney?</b></summary>

The t-test compares means and assumes roughly normal sample means; Mann–Whitney compares ranks and is robust to outliers and skew, at a small cost in power for normal data.

</details>

<details>
<summary><b>6. What is the multiple comparisons problem?</b></summary>

Running many tests makes some false positives almost certain. Correct with Bonferroni (controls any false positive) or Benjamini–Hochberg (controls the false discovery rate).

</details>

<details>
<summary><b>7. Statistical vs practical significance?</b></summary>

Statistical significance says the effect is probably not zero; practical significance says it is big enough to matter. Always report effect sizes and confidence intervals.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: The logic of a hypothesis test
- [ ] I can explain: Two kinds of mistakes
- [ ] I can explain: Which test? z, t, paired, Welch
- [ ] I can explain: Categories and many groups: χ² and ANOVA
- [ ] I can explain: When formulas don't fit: ranks, permutations, the bootstrap
- [ ] I can explain: Significant is not the same as important

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Mann–Whitney U
- [ ] [Q17](#q17) Exact Binomial Test
- [ ] [Q18](#q18) How Many Users Does the A/B Test Need?
- [ ] [Q19](#q19) Permutation Test
- [ ] [Q20](#q20) False Discovery Rate (Benjamini–Hochberg)

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.